In [1]:
import math

In [2]:
# Nạp thư viện và cấu hình chung
TRAFFIC_CONFIG = {
    'average_speed': {
        'motorbike': 35,
        'car': 30,
        'walking': 4.5,
    },
    'road_distance_factor': {
        'motorbike': 1.2,
        'car': 1.3,
        'walking': 1.05,
    },
}

In [3]:
# Hàm tính khoảng cách đường chim bay giữa 2 tọa độ

def calculate_air_distance_km(start_lat, start_lng, end_lat, end_lng):
    earth_radius_km = 6371.0

    start_lat = float(start_lat)
    start_lng = float(start_lng)
    end_lat = float(end_lat)
    end_lng = float(end_lng)

    delta_lat = math.radians(end_lat - start_lat)
    delta_lng = math.radians(end_lng - start_lng)

    a = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(math.radians(start_lat))
        * math.cos(math.radians(end_lat))
        * math.sin(delta_lng / 2) ** 2
    )
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return earth_radius_km * c

In [4]:
# tính quãng đường ước lượng theo chế độ di chuyển

def estimate_road_distance_km(air_distance_km, delivery_mode='motorbike'):
    road_factor = TRAFFIC_CONFIG['road_distance_factor'].get(delivery_mode, 1.2)
    return air_distance_km * road_factor

In [5]:
# tính thời gian di chuyển dự kiến

def estimate_travel_time_minutes(estimated_distance_km, delivery_mode='motorbike'):
    average_speed = TRAFFIC_CONFIG['average_speed'].get(delivery_mode, 30)

    if average_speed <= 0:
        return 1

    estimated_minutes = int(round((estimated_distance_km / average_speed) * 60))
    return max(estimated_minutes, 1)

In [6]:
# tính phí ship theo quãng đường ước lượng

def calculate_shipping_fee(estimated_distance_km):
    try:
        distance_value = float(estimated_distance_km)
    except (TypeError, ValueError):
        distance_value = 0.0

    if distance_value <= 3:
        shipping_fee_value = 15000
    else:
        shipping_fee_value = 15000 + int((distance_value - 3) * 5000)

    shipping_fee_value = int(round(shipping_fee_value, -3))
    shipping_fee_text = f"{shipping_fee_value:,} đ".replace(",", ".")

    return shipping_fee_value, shipping_fee_text

In [7]:
# Hàm tính toàn bộ kết quả giao hàng cho một nhà thuốc


def estimate_route_result(pharmacy_lat, pharmacy_lng, delivery_lat, delivery_lng, delivery_mode='motorbike'):
    air_distance_km = calculate_air_distance_km(
        pharmacy_lat,
        pharmacy_lng,
        delivery_lat,
        delivery_lng,
    )

    estimated_road_distance_km = estimate_road_distance_km(
        air_distance_km,
        delivery_mode,
    )

    duration_min = estimate_travel_time_minutes(
        estimated_road_distance_km,
        delivery_mode,
    )

    shipping_fee_value, shipping_fee_text = calculate_shipping_fee(
        estimated_road_distance_km,
    )

    return {
        'delivery_mode': delivery_mode,
        'estimated_road_distance_km': round(estimated_road_distance_km, 2),
        'duration_min': duration_min,
        'shipping_fee_text': shipping_fee_text,
    }

In [8]:
# chọn nhà thuốc tối ưu
# Tiêu chí chọn:
# - nhà thuốc có quãng đường ước lượng ngắn nhất

def choose_best_pharmacy(pharmacies, delivery_lat, delivery_lng, delivery_mode='motorbike'):
    best_result = None
    shortest_distance_km = None

    for pharmacy in pharmacies:
        route_result = estimate_route_result(
            pharmacy_lat=pharmacy['lat'],
            pharmacy_lng=pharmacy['lng'],
            delivery_lat=delivery_lat,
            delivery_lng=delivery_lng,
            delivery_mode=delivery_mode,
        )

        current_distance_km = route_result['estimated_road_distance_km']

        if shortest_distance_km is None or current_distance_km < shortest_distance_km:
            shortest_distance_km = current_distance_km
            best_result = {
                'pharmacy_id': pharmacy['id'],
                'pharmacy_name': pharmacy['name'],
                'route_result': route_result,
            }

    return best_result

In [9]:
# Nạp dữ liệu đầu vào
# Dữ liệu gồm:
# - danh sách nhà thuốc
# - vị trí giao hàng
# - chế độ di chuyển

pharmacies = [
    {'id': 1, 'name': 'Nhà thuốc A', 'lat': 10.7765, 'lng': 106.7009},
    {'id': 2, 'name': 'Nhà thuốc B', 'lat': 10.7815, 'lng': 106.6952},
    {'id': 3, 'name': 'Nhà thuốc C', 'lat': 10.7680, 'lng': 106.7055},
]

delivery_lat = 10.7721
delivery_lng = 106.6983
delivery_mode = 'motorbike'

In [10]:
# Gọi hàm để tính kết quả cho từng nhà thuốc

for pharmacy in pharmacies:
    result = estimate_route_result(
        pharmacy_lat=pharmacy['lat'],
        pharmacy_lng=pharmacy['lng'],
        delivery_lat=delivery_lat,
        delivery_lng=delivery_lng,
        delivery_mode=delivery_mode,
    )

    print(pharmacy['name'])
    print(result)
    print("-" * 60)

Nhà thuốc A
{'delivery_mode': 'motorbike', 'estimated_road_distance_km': 0.68, 'duration_min': 1, 'shipping_fee_text': '15.000 đ'}
------------------------------------------------------------
Nhà thuốc B
{'delivery_mode': 'motorbike', 'estimated_road_distance_km': 1.32, 'duration_min': 2, 'shipping_fee_text': '15.000 đ'}
------------------------------------------------------------
Nhà thuốc C
{'delivery_mode': 'motorbike', 'estimated_road_distance_km': 1.09, 'duration_min': 2, 'shipping_fee_text': '15.000 đ'}
------------------------------------------------------------


In [11]:
# Gọi hàm chọn nhà thuốc tối ưu và in kết quả 

best_pharmacy_result = choose_best_pharmacy(
    pharmacies=pharmacies,
    delivery_lat=delivery_lat,
    delivery_lng=delivery_lng,
    delivery_mode=delivery_mode,
)

print("Nhà thuốc được chọn:")
print(best_pharmacy_result)

Nhà thuốc được chọn:
{'pharmacy_id': 1, 'pharmacy_name': 'Nhà thuốc A', 'route_result': {'delivery_mode': 'motorbike', 'estimated_road_distance_km': 0.68, 'duration_min': 1, 'shipping_fee_text': '15.000 đ'}}
